In [1]:
import genesis as gs

gs.init(backend=gs.metal, logging_level="warning")

[I 12/06/25 21:54:42.755 200801204] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


In [ ]:
from turtle import pos
import numpy as np

########################## create a scene ##########################
scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, -3.5, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=60,
    ),
    show_viewer=True,
)

########################## entities ##########################
plane = scene.add_entity(
    gs.morphs.Plane(),
)

# when loading an entity, you can specify its pose in the morph.
franka = scene.add_entity(
    gs.morphs.MJCF(
        file="xml/franka_emika_panda/panda.xml",
        pos=(1.0, 1.0, 0.0),
        euler=(0, 0, 0),
    ),
)

cube = scene.add_entity(
    gs.morphs.Box(
        pos=(-0.4, 0.0, 0),
        size=(0.05, 0.05, 0.05),
    )
)
########################## build ##########################
scene.build()
############ Optional: set control gains ############
jnt_names = [
    "joint1",
    "joint2",
    "joint3",
    "joint4",
    "joint5",
    "joint6",
    "joint7",
    "finger_joint1",
    "finger_joint2",
]
dofs_idx = [franka.get_joint(name).dof_idx_local for name in jnt_names]
# set positional gains
franka.set_dofs_kp(
    kp=np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]),
    dofs_idx_local=dofs_idx,
)
# set velocity gains
franka.set_dofs_kv(
    kv=np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]),
    dofs_idx_local=dofs_idx,
)
# set force range for safety
franka.set_dofs_force_range(
    lower=np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    upper=np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
    dofs_idx_local=dofs_idx,
)

# PD control
# for i in range(1250):
#     if i == 0:
#         franka.control_dofs_position(
#             np.array([1, 1, 0, 0, 0, 0, 0, 0.04, 0.04]),
#             dofs_idx,
#         )
#     elif i == 250:
#         franka.control_dofs_position(
#             np.array([-1, 0.8, 1, -2, 1, 0.5, -0.5, 0.04, 0.04]),
#             dofs_idx,
#         )
#     elif i == 500:
#         franka.control_dofs_position(
#             np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
#             dofs_idx,
#         )
#     elif i == 750:
#         # control first dof with velocity, and the rest with position
#         franka.control_dofs_position(
#             np.array([0, 0, 0, 0, 0, 0, 0, 0, 0])[1:],
#             dofs_idx[1:],
#         )
#         franka.control_dofs_velocity(
#             np.array([1.0, 0, 0, 0, 0, 0, 0, 0, 0])[:1],
#             dofs_idx[:1],
#         )
#     elif i == 1000:
#         franka.control_dofs_force(
#             np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
#             dofs_idx,
#         )
#     # This is the control force computed based on the given control command
#     # If using force control, it's the same as the given control command
#     print('control force:', franka.get_dofs_control_force(dofs_idx))

#     # This is the actual force experienced by the dof
#     print('internal force:', franka.get_dofs_force(dofs_idx))

#     scene.step()

# --- (scene and robot creation omitted, identical to the sections above) ---

# Retrieve some commonly used handles
rigid = scene.sim.rigid_solver  # low-level rigid body solver
end_effector = franka.get_link("hand")  # Franka gripper frame
cube_link = cube.get_link("box_baselink")  # the link we want to pick

################ Reach pre-grasp pose ################
q_pregrasp = franka.inverse_kinematics(
    link=end_effector,
    pos=np.array([0.65, 0.0, 0.13]),  # just above the cube
    quat=np.array([0, 1, 0, 0]),  # down-facing orientation
)
franka.control_dofs_position(q_pregrasp, np.arange(9))  # arm joints only
for _ in range(50):
    scene.step()

################ Attach (activate suction) ################
link_cube = np.array([cube_link.idx], dtype=gs.np_int)
link_franka = np.array([end_effector.idx], dtype=gs.np_int)
rigid.add_weld_constraint(link_cube, link_franka)

################ Lift and transport ################
q_lift = franka.inverse_kinematics(
    link=end_effector,
    pos=np.array([0.65, 0.0, 0.28]),  # lift up
    quat=np.array([0, 1, 0, 0]),
)
franka.control_dofs_position(q_lift[:-2], np.arange(7))
for _ in range(50):
    scene.step()

q_place = franka.inverse_kinematics(
    link=end_effector,
    pos=np.array([0.4, 0.2, 0.18]),  # target place pose
    quat=np.array([0, 1, 0, 0]),
)
franka.control_dofs_position(q_place[:-2], np.arange(7))
for _ in range(100):
    scene.step()

################ Detach (release suction) ################
rigid.delete_weld_constraint(link_cube, link_franka)
for _ in range(400):
    scene.step()

scene.destroy()

In [2]:
scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.02),
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, -3.5, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=30,
    ),
    show_viewer=True,
)
scene.add_entity(gs.morphs.Plane())
robot = scene.add_entity(
    gs.morphs.MJCF(
        file="/Users/nkayslaptop/Desktop/Master's Program/Reinforcement learning/Final Project/GPT-Reward/robots/boston_dynamics_spot/spot.xml",
        pos=(0, 0, 0),
        quat=(1, 0, 0, 0),
    ),
)
scene.build(n_envs=1)

dof_names = [
            "fl_hx",
            "fr_hx",
            "hl_hx",
            "hr_hx",
            "fl_hy",
            "fr_hy",
            "hl_hy",
            "hr_hy",
            "fl_kn",
            "fr_kn",
            "hl_kn",
            "hr_kn",
        ]
dofs_idx = [
    robot.get_joint(name).dof_idx_local for name in dof_names
]

robot.set_dofs_kp([1000] * 12, dofs_idx)
robot.set_dofs_kv([50] * 12, dofs_idx)
robot.set_dofs_force_range(
    lower=[-100] * 12,
    upper=[100] * 12,
    dofs_idx_local=dofs_idx,
)

import numpy as np
for i in range(400):
    if i == 0:
        robot.control_dofs_position(
            np.array([0, 0, 0, 0, 0.7, 0.7, 0.7, 0.7, -1.3, -1.3, -1.3, -1.3]),
            dofs_idx,
        )
    scene.step()
print(robot.get_pos())
scene.destroy()

[Genesis] [21:54:49] [WARNING] Interactive viewer running in main thread. It will only be responsive if a simulation is running.
[Genesis] [21:54:51] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.004` to `0.04`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [21:54:51] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.02` to `0.04`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [21:54:51] [WARNING] Reference robot position exceeds joint limits.


UNSUPPORTED (log once): POSSIBLE ISSUE: unit 4 GLD_TEXTURE_INDEX_CUBE_MAP is unloadable and bound to sampler type (Float) - using zero texture because texture unloadable


[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:55:30] [WARNING] This property is depreca

In [4]:
scene.destroy()

In [5]:
robot.get_dofs_velocity()

tensor([[ 1.9587e-07,  1.0316e-06, -6.3528e-07, -1.0880e-06,  1.8097e-06,
          4.6434e-07, -6.8139e-07, -7.9394e-07,  3.7286e-07, -9.0284e-08,
         -8.4700e-07, -1.9518e-08,  9.7954e-07, -1.2147e-06, -1.3961e-06,
         -2.2568e-06, -5.4151e-06,  7.9974e-07]], device='mps:0')

In [6]:
robot.get_vel()

tensor([[ 1.9587e-07,  1.0316e-06, -6.3528e-07]], device='mps:0')

In [7]:
robot.get_ang()

tensor([[-1.0965e-06,  1.8052e-06,  4.6164e-07]], device='mps:0')

In [ ]:
from genesis.utils.geom import quat_to_xyz,quat_to_R

quat_to_R(robot.get_quat())

tensor([[[ 9.9999e-01, -4.0822e-03, -2.5201e-03],
         [ 4.0821e-03,  9.9999e-01, -3.1236e-05],
         [ 2.5202e-03,  2.0949e-05,  1.0000e+00]]], device='mps:0')

: 

In [15]:
robot.get_pos()

tensor([[0.0015, 0.0015, 0.5546]], device='mps:0')

In [14]:
robot.get_qpos()

tensor([[ 1.4659e-03,  1.5317e-03,  5.5456e-01,  1.0000e+00,  1.3046e-05,
         -1.2601e-03,  2.0411e-03,  1.0038e-02, -1.0049e-02,  1.2572e-02,
         -1.2582e-02,  6.9667e-01,  6.9667e-01,  6.9624e-01,  6.9624e-01,
         -1.3220e+00, -1.3220e+00, -1.3262e+00, -1.3262e+00]], device='mps:0')

In [ ]:
import torch

torch.tensor([1,2,3]).scatt

(12, 2)

In [159]:
ranges[:, 0].reshape(1,12).repeat(2, axis=0)

array([[-0.785398, -0.785398, -0.785398, -0.785398, -0.898845, -0.898845,
        -0.898845, -0.898845, -2.7929  , -2.7929  , -2.7929  , -2.7929  ],
       [-0.785398, -0.785398, -0.785398, -0.785398, -0.898845, -0.898845,
        -0.898845, -0.898845, -2.7929  , -2.7929  , -2.7929  , -2.7929  ]])

In [128]:
help(robot.zero_all_dofs_velocity)

Help on method zero_all_dofs_velocity in module genesis.engine.entities.rigid_entity.rigid_entity:

zero_all_dofs_velocity(envs_idx=None, *, unsafe=False) method of genesis.engine.entities.rigid_entity.rigid_entity.RigidEntity instance
    Zero the velocity of all the entity's dofs.
    
    Parameters
    ----------
    envs_idx : None | array_like, optional
        The indices of the environments. If None, all environments will be considered. Defaults to None.



In [ ]:
v = np.array([1.0, 0.0, 0.0, 0.0])
v.reshape(1, -1)

array([[1., 0., 0., 0.]])

In [67]:
v

array([1., 0., 0., 0.])

In [3]:
import torch

torch.ones((3,))

tensor([1., 1., 1.], device='mps:0')

In [71]:
help(robot.get_vel)

Help on method get_vel in module genesis.engine.entities.rigid_entity.rigid_entity:

get_vel(envs_idx=None, *, unsafe=False) method of genesis.engine.entities.rigid_entity.rigid_entity.RigidEntity instance
    Returns linear velocity of the entity's base link.
    
    Parameters
    ----------
    envs_idx : None | array_like, optional
        The indices of the environments. If None, all environments will be considered. Defaults to None.
    
    Returns
    -------
    vel : torch.Tensor, shape (3,) or (n_envs, 3)
        The linear velocity of the entity's base link.



In [70]:
help(robot.get_dofs_velocity)

Help on method get_dofs_velocity in module genesis.engine.entities.rigid_entity.rigid_entity:

get_dofs_velocity(dofs_idx_local=None, envs_idx=None, *, unsafe=False) method of genesis.engine.entities.rigid_entity.rigid_entity.RigidEntity instance
    Get the entity's dofs' velocity.
    
    Parameters
    ----------
    dofs_idx_local : None | array_like, optional
        The indices of the dofs to get. If None, all dofs will be returned. Note that here this uses the local `q_idx`, not the scene-level one. Defaults to None.
    envs_idx : None | array_like, optional
        The indices of the environments. If None, all environments will be considered. Defaults to None.
    
    Returns
    -------
    velocity : torch.Tensor, shape (n_dofs,) or (n_envs, n_dofs)
        The entity's dofs' velocity.



In [ ]:
from genesis.utils.geom import (
    quat_to_xyz,
    transform_by_quat,
    inv_quat,
    transform_quat_by_quat,
)

help(inv_quat)

Help on function inv_quat in module genesis.utils.geom:

inv_quat(quat)



In [73]:
help(robot.get_vel)

Help on method get_vel in module genesis.engine.entities.rigid_entity.rigid_entity:

get_vel(envs_idx=None, *, unsafe=False) method of genesis.engine.entities.rigid_entity.rigid_entity.RigidEntity instance
    Returns linear velocity of the entity's base link.
    
    Parameters
    ----------
    envs_idx : None | array_like, optional
        The indices of the environments. If None, all environments will be considered. Defaults to None.
    
    Returns
    -------
    vel : torch.Tensor, shape (3,) or (n_envs, 3)
        The linear velocity of the entity's base link.



In [74]:
help(transform_by_quat)

Help on function transform_by_quat in module genesis.utils.geom:

transform_by_quat(v, quat)
    This method transforms quat_v by quat_u.
    
    This is equivalent to quatmul(quat_u, quat_v) or R_u @ R_v



In [ ]:
import torch

torch.cat([torch.tensor([[1, 2, 3]]), torch.tensor([[4, 5, 6]])], axis=-1)

tensor([[1, 2, 3, 4, 5, 6]], device='mps:0')

In [ ]:
from torch import tensor

[
    tensor([[0.0, 0.0, 0.0]], device="mps:0"),
    tensor([[0.0, 0.0, -1.0]], device="mps:0"),
    tensor([[0.0000, 0.0000, 0.0000, 0.6000, 0.0000]], device="mps:0"),
    tensor(
        [
            [
                0.0186,
                0.0057,
                0.0192,
                0.0068,
                0.0125,
                0.0129,
                0.0120,
                0.0124,
                -0.0580,
                -0.0556,
                -0.0564,
                -0.0540,
            ]
        ],
        device="mps:0",
    ),
    tensor(
        [
            [
                0.0466,
                0.0141,
                0.0479,
                0.0170,
                0.0311,
                0.0322,
                0.0299,
                0.0309,
                -0.1451,
                -0.1390,
                -0.1411,
                -0.1350,
            ]
        ],
        device="mps:0",
    ),
    tensor(
        [
            0.7214,
            -0.2668,
            -0.7558,
            0.5378,
            0.5199,
            1.5534,
            0.3072,
            -0.6853,
            -2.5051,
            -2.3842,
            -2.4903,
            -0.3891,
        ],
        device="mps:0",
    ),
    tensor([[0.0]], device="mps:0"),
]

In [ ]:
[
    tensor([[0.0, 0.0, 0.0]], device="mps:0"),
    tensor([[0.0, 0.0, -1.0]], device="mps:0"),
    tensor([[0.0000, 0.0000, 0.0000, 0.6000, 0.0000]], device="mps:0"),
    tensor(
        [
            [
                0.0186,
                0.0057,
                0.0192,
                0.0068,
                0.0125,
                0.0129,
                0.0120,
                0.0124,
                -0.0580,
                -0.0556,
                -0.0564,
                -0.0540,
            ]
        ],
        device="mps:0",
    ),
    tensor(
        [
            [
                0.0466,
                0.0141,
                0.0479,
                0.0170,
                0.0311,
                0.0322,
                0.0299,
                0.0309,
                -0.1451,
                -0.1390,
                -0.1411,
                -0.1350,
            ]
        ],
        device="mps:0",
    ),
    tensor(
        [
            [
                -0.4060,
                -0.3017,
                0.6687,
                -0.4358,
                0.1958,
                0.1514,
                -0.8405,
                -0.4078,
                -2.0761,
                -1.6632,
                -0.8008,
                -0.7703,
            ]
        ],
        device="mps:0",
    ),
    tensor([[0.0]], device="mps:0"),
]